# RNN

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_recurrent-neural-networks/rnn-concise.ipynb` · [Lección original](https://d2l.ai/chapter_recurrent-neural-networks/rnn-concise.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Implementación concisa de redes neuronales recurrentes
<a id="sec_rnn-concise"></a>

Como la mayoría de nuestras implementaciones de arañazos,
[Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch) se diseñó 
Pero cuando esté usando RNNs todos los días o escribiendo código de producción, querrá confiar más en bibliotecas que reduzcan el tiempo de implementación (proporcionando código de biblioteca para modelos y funciones comunes) y el tiempo de computación (optimizando las implementaciones de estas bibliotecas). Esta sección le mostrará cómo implementar el mismo modelo de lenguaje de manera más eficiente utilizando la API de alto nivel proporcionada por su biblioteca de aprendizaje profundo. Comenzamos, como antes, cargando el conjunto de datos *The Time Machine*.


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from laboratorio import d2l

## Definición del modelo

Definimos la siguiente clase utilizando la RNN implementada por API de alto nivel.


In [ ]:
class RNN(d2l.Module):  #@save
    """El modelo RNN se implementó con API de alto nivel."""
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.save_hyperparameters()
        self.rnn = nn.RNN(num_inputs, num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

Heredera de la clase `RNNLMScratch` en [Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch), la siguiente clase `RNNLM` define un modelo completo de lenguaje basado en RNN. Tenga en cuenta que necesitamos crear una capa de salida separada totalmente conectada.


In [ ]:
class RNNLM(d2l.RNNLMScratch):  #@save
    """El modelo de lenguaje basado en RNN implementado con API de alto nivel."""
    def init_params(self):
        self.linear = nn.LazyLinear(self.vocab_size)

    def output_layer(self, hiddens):
        return self.linear(hiddens).swapaxes(0, 1)

## Entrenamiento y predicción
Antes de entrenar el modelo, **hagamos una predicción con un modelo inicializado con pesos aleatorios.** Dado que no hemos entrenado la red, generará predicciones sin sentido.


### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «RNN».


In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)
rnn = RNN(num_inputs=len(data.vocab), num_hiddens=32)
model = RNNLM(rnn, vocab_size=len(data.vocab), lr=1)
model.predict('it has', 20, data.vocab)

A continuación, **entrenamos nuestro modelo, aprovechando la API de alto nivel**.


In [ ]:
trainer = d2l.Trainer(max_epochs=100, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

Comparado con [Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch), este modelo logra una perplejidad comparable, pero se ejecuta más rápido debido a las implementaciones optimizadas. Como antes, podemos generar tokens predichos siguiendo la cadena de prefijo especificada.


In [ ]:
model.predict('it has', 20, data.vocab, d2l.try_gpu())

## Resumen
Las API de alto nivel en bibliotecas de aprendizaje profundo proporcionan implementaciones de RNN estándar. Estas bibliotecas le ayudan a evitar perder tiempo reimplementando modelos estándar. Además, las implementaciones de marcos a menudo están altamente optimizadas, lo que conduce a ganancias significativas (computacionales) de rendimiento en comparación con las implementaciones desde cero.

## Ejercicios
1. ¿Puede hacer que el modelo RNN se ajuste de más utilizando las API de alto nivel?
1. Implementar el modelo autorregresivo de [Referencia sec_sequence](https://d2l.ai/chapter_recurrent-neural-networks/sequence.html#sec-sequence) usando un RNN.


[Debate del original](https://discuss.d2l.ai/t/1053)
